# Prediction Module

Generates predictions for **unseen data** using the model trained in `model_training.ipynb`.

Workflow: **Upload data -> Load saved model -> Generate predictions -> Download results**

This notebook is dataset-agnostic — it relies entirely on the artifacts saved during
training (`models/best_model.pkl`, `models/preprocessing_artifacts.pkl`) to know which
raw columns, encoders, and scaler to apply. Point it at any new file with the same raw
schema and it works without editing any code below.

In [1]:
import sys, os, io, base64
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import ipywidgets as widgets
from IPython.display import display, HTML

src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.append(src_path)

%load_ext autoreload
%autoreload 2

from prediction import load_artifacts, generate_predictions, save_predictions

print("Setup complete. src in path:", src_path in sys.path)

Setup complete. src in path: True


## 1. Load the saved model

Loads `best_model.pkl` (the trained model chosen by `model_training.ipynb`) together
with `preprocessing_artifacts.pkl` (the encoders, scaler and feature list fitted during
training). Both are required so unseen data can be transformed exactly the same way.

In [2]:
MODELS_DIR = os.path.abspath("../models")

model_bundle, artifacts = load_artifacts(MODELS_DIR)

print(f"Model             : {model_bundle['model_name']}")
print(f"Target column     : {model_bundle['target_col']}")
print(f"Features expected : {len(model_bundle['feature_columns'])}")
print(f"Validation metrics: {model_bundle['metrics']}")

2026-07-09 17:33:34,813 | prediction | INFO | Loaded model 'RandomForest' and preprocessing artifacts (15 features).


Model             : RandomForest
Target column     : diabetes
Features expected : 15
Validation metrics: {'Accuracy': 0.9639, 'Precision': 0.9632, 'Recall': 0.9639, 'F1 Score': 0.9635, 'ROC-AUC': 0.9716}


## 2. Upload unseen data

Upload a CSV or Excel file containing the same raw columns used at training time
(the target column is optional — include it if you have it, it will simply be ignored;
omit it if you don't, since that's the whole point of predicting on unseen data).

In [3]:
upload_widget = widgets.FileUpload(
    accept=".csv,.xlsx,.xls",
    multiple=False,
    description="Upload data",
)
upload_output = widgets.Output()

state = {"raw_df": None, "results_df": None}


def _extract_uploaded_file(value):
    # Handle both the ipywidgets 7.x (dict) and 8.x (tuple) FileUpload.value shapes.
    if isinstance(value, dict):
        item = next(iter(value.values()))
        return item["metadata"]["name"], bytes(item["content"])
    item = value[0]
    return item["name"], bytes(item["content"])


def _read_uploaded_file(name, content_bytes):
    buffer = io.BytesIO(content_bytes)
    if name.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(buffer)
    return pd.read_csv(buffer)


def on_upload_change(change):
    upload_output.clear_output()
    if not upload_widget.value:
        return
    name, content_bytes = _extract_uploaded_file(upload_widget.value)
    df = _read_uploaded_file(name, content_bytes)
    state["raw_df"] = df
    state["results_df"] = None
    with upload_output:
        print(f"Loaded '{name}' -> {df.shape[0]} rows x {df.shape[1]} columns")
        display(df.head())


upload_widget.observe(on_upload_change, names="value")
display(upload_widget, upload_output)

FileUpload(value=(), accept='.csv,.xlsx,.xls', description='Upload data')

Output()

## 3. Generate predictions

Applies the saved cleaning, encoding and scaling to the uploaded data, then runs the
saved model. Missing expected columns are filled with 0, unexpected extra columns are
dropped, and unseen category values are mapped to a safe default — all handled
dynamically, so no code changes are needed for a new file.

In [4]:
predict_button = widgets.Button(description="Generate Predictions", button_style="primary", icon="play")
predict_output = widgets.Output()


def on_predict_click(_):
    predict_output.clear_output()
    with predict_output:
        if state["raw_df"] is None:
            print("Please upload a data file first (step 2).")
            return
        try:
            results_df, prep_report = generate_predictions(state["raw_df"], model_bundle, artifacts)
        except Exception as e:
            print(f"Prediction failed: {e}")
            return

        state["results_df"] = results_df

        if prep_report["missing_raw_columns"]:
            print(f"Note: columns expected by the model but missing from upload (filled with 0): "
                  f"{prep_report['missing_raw_columns']}")
        if prep_report["extra_raw_columns"]:
            print(f"Note: extra columns in upload not used by the model: {prep_report['extra_raw_columns']}")
        if prep_report["unseen_categories"]:
            print(f"Note: unseen category values encountered (mapped to a default): "
                  f"{prep_report['unseen_categories']}")

        pred_col = "predicted_" + model_bundle["target_col"]
        print(f"\nGenerated {len(results_df)} predictions with '{model_bundle['model_name']}'.")
        print(results_df[pred_col].value_counts())
        display(results_df.head(10))


predict_button.on_click(on_predict_click)
display(predict_button, predict_output)

Button(button_style='primary', description='Generate Predictions', icon='play', style=ButtonStyle())

Output()

## 4. Download prediction results

Saves the predictions to `reports/` and provides a direct browser download link.

In [5]:
filename_widget = widgets.Text(
    value=f"{model_bundle['target_col']}_predictions.csv",
    description="File name:",
    style={"description_width": "initial"},
)
download_button = widgets.Button(description="Download Predictions", button_style="success", icon="download")
download_output = widgets.Output()


def create_download_link(df, filename):
    csv_bytes = df.to_csv(index=False).encode()
    b64 = base64.b64encode(csv_bytes).decode()
    return HTML(f'<a download="{filename}" href="data:text/csv;base64,{b64}" target="_blank">'
                f'Click to download {filename}</a>')


def on_download_click(_):
    download_output.clear_output()
    with download_output:
        results_df = state.get("results_df")
        if results_df is None:
            print("Generate predictions first (step 3).")
            return
        filename = filename_widget.value.strip() or "predictions.csv"
        saved_path = save_predictions(results_df, reports_dir="../reports", filename=filename)
        print(f"Saved -> {saved_path}")
        display(create_download_link(results_df, filename))


download_button.on_click(on_download_click)
display(filename_widget, download_button, download_output)

Text(value='diabetes_predictions.csv', description='File name:', style=TextStyle(description_width='initial'))

Button(button_style='success', description='Download Predictions', icon='download', style=ButtonStyle())

Output()